In [ ]:
import sys

In [ ]:
sys.path.insert(0, "/cmmc/ptmp/zora/software/assyst-nodes")
sys.path.insert(0, "/cmmc/ptmp/zora/software/pyiron_core.pyiron_workflow")
sys.path.insert(0, "/cmmc/ptmp/zora/software/pyironflow")

In [ ]:
from aiflow.core import Workflow

In [ ]:
from assyst_nodes.assyst.calculators import (
    GenericOptimizerSettings,
    M3gnetConfig,
    Relax,
)
from assyst_nodes.assyst.plot import PlotSPG
from assyst_nodes.assyst.structures import SpaceGroupInput, SpaceGroupSampling


def make_assyst(name, delete_existing_savefiles=False):
    wf = Workflow(name, delete_existing_savefiles=delete_existing_savefiles)
    if wf.has_saved_content():
        return wf
    inp = SpaceGroupInput()
    spg = SpaceGroupSampling(inp)
    plotspg = PlotSPG(spg)

    calc_config = M3gnetConfig()
    optimizer_settings = GenericOptimizerSettings()

    volume_relax = Relax().for_node(
        mode="volume",
        calculator=calc_config,
        opt=optimizer_settings,
        structure=spg,
        iter_on="structure",
        output_as_dataframe=False,
    )
    full_relax = Relax().for_node(
        mode="full",
        calculator=calc_config,
        opt=optimizer_settings,
        iter_on="structure",
        output_as_dataframe=False,
    )
    full_relax.inputs.structure = volume_relax.outputs.relaxed_structure

    wf.input = inp
    wf.sampling = spg
    wf.plotspg = plotspg

    wf.calc_config = calc_config
    wf.optimizer_settings = optimizer_settings
    wf.volume_relax = volume_relax
    wf.full_relax = full_relax

    wf.inputs_map = {
        "input__elements": "elements",
        "input__max_atoms": "max_atoms",
        "input__stoichiometry": "stoichiometry",
        "input__spacegroups": "spacegroups",
        "input__min_dist": "min_dist",
        "calc_config__model": None,
        "volume_relax__mode": None,
        "full_relax__mode": None,
    }
    wf.outputs_map = {
        "sampling__structures": "crystals",
        "volume_relax__relaxed_structure": "volmin",
        "full_relax__relaxed_structure": "allmin",
        "volume_relax__structure": None,
        "full_relax__structure": None,
    }

    return wf

In [ ]:
wf = make_assyst("ASSYST", delete_existing_savefiles=True)

In [ ]:
wf.inputs.elements = ["Mg", "Ca"]
wf.inputs.stoichiometry = [(2, 1)]

In [ ]:
wf.run()

In [ ]:
wf.draw(size=(16, 8))

In [ ]:
wf.save()

In [ ]:
wf2 = Workflow("ASSYST")

Saving and reloading works, but somehow not with the gui.

In [ ]:
from pyiron_core.pyironflow import PyironFlow

In [ ]:
pf = PyironFlow([wf2])

In [ ]:
pf = PyironFlow([wf])

In [ ]:
pf.gui